In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn import metrics

# ==========================================
# EXERCISE 1 - Data Collection & Exploration
# ==========================================

# TODO: load the dataset
# Remplacez 'diabetes_prediction_dataset.csv' par le chemin exact de votre fichier
df = pd.read_csv('diabetes_prediction_dataset.csv')

print("Shape of dataset:", df.shape)
display(df.head())
print("\nData types:\n", df.dtypes)
print("\nMissing per column:")
display(df.isna().sum().sort_values(ascending=False))

# Target column verification
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"
print("\nTarget counts (0 = Negative, 1 = Positive):")
print(df['diabetes'].value_counts())

# TODO: train test split
X = df.drop(columns=['diabetes'])
y = df['diabetes']

# Split 80% train / 20% test avec stratification pour préserver le ratio de classes
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\nTrain and Test shapes:", X_train.shape, X_test.shape)


# ==========================================
# EXERCISE 2 - Justification & Preprocessing
# ==========================================

"""
JUSTIFICATION (A copier dans la cellule Markdown prévue à cet effet) :
La régression logistique est parfaitement adaptée à cette tâche de classification binaire car elle modélise directement la probabilité conditionnelle P(y=1|X) à l'aide d'une fonction sigmoïde, offrant ainsi des probabilités calibrées et une frontière de décision linéaire très robuste. De plus, elle est hautement interprétable grâce à l'analyse de ses coefficients (odds ratios).
La standardisation des caractéristiques numériques à l'aide de StandardScaler est indispensable ici : elle garantit une meilleure stabilité numérique lors de la descente de gradient, accélère la convergence de l'optimiseur et empêche les variables à forte variance d'écraser artificiellement les autres caractéristiques.
"""

# TODO: build a preprocessing pipeline
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

print("\nCategorical columns:", cat_cols)
print("Numeric columns:", num_cols)


# ==========================================
# EXERCISE 3 - Model Training
# ==========================================

# TODO: train Logistic Regression
clf = Pipeline([
    ('preprocessor', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(X_train, y_train)
print("\nModèle entraîné avec succès !")


# ==========================================
# EXERCISE 4 - Evaluation Metrics
# ==========================================

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n--- Evaluation Metrics ---")
print("Accuracy:", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall:", round(rec, 4))
print("F1:", round(f1, 4))

# Simple bar plot of metrics
plt.figure(figsize=(7, 4))
plt.bar(['accuracy','precision','recall','f1'], [acc, prec, rec, f1], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Metrics on test set')
plt.ylabel('Score')
plt.ylim(0, 1.1)
for i, v in enumerate([acc, prec, rec, f1]):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap='Blues')
plt.title('Confusion matrix')
plt.show()

"""
COMMENTAIRE SUR L'ÉQUILIBRE PRECISION/RECALL (A ajouter en Markdown) :
L'analyse des scores montre généralement (sur ce dataset) une excellente 'Precision' (les cas prédits positifs sont très souvent de vrais diabétiques), mais un 'Recall' (sensibilité) légèrement inférieur. Dans un contexte médical, un faible Recall signifie que le modèle laisse passer de faux négatifs (des personnes malades non détectées). Il pourrait être intéressant d'ajuster le seuil de décision en dessous de 0.5 pour capturer plus de cas positifs au détriment d'un peu de précision.
"""


# ==========================================
# EXERCISE 5 - 2D Decision Boundary
# ==========================================

# Sélection des variables d'intérêt
feat_x = 'HbA1c_level' if 'HbA1c_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test = X_test[[feat_x, feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0,1])], remainder='drop')),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Meshgrid pour les contours
x_min, x_max = X2_train[feat_x].min() - 1, X2_train[feat_x].max() + 1
y_min, y_max = X2_train[feat_y].min() - 1, X2_train[feat_y].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

plt.figure(figsize=(7, 6))
cs = plt.contour(xx, yy, probs, levels=[0.5], colors='red', linewidths=2)
plt.clabel(cs, inline=True, fmt={0.5: 'Seuil P=0.5'}, fontsize=12)

# Affichage des points de test (échantillonné pour la lisibilité si le dataset est trop grand)
sample_idx = np.random.choice(len(X2_test), min(1000, len(X2_test)), replace=False)
plt.scatter(X2_test[feat_x].iloc[sample_idx], X2_test[feat_y].iloc[sample_idx], c=y_test.iloc[sample_idx], alpha=0.6, cmap='coolwarm', edgecolors='k', s=25)

plt.xlabel(feat_x)
plt.ylabel(feat_y)
acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))
plt.title(f'Decision boundary on 2 features - test accuracy {acc2:.3f}')
plt.colorbar(label="Classe (0: Sain, 1: Diabétique)")
plt.show()


# ==========================================
# EXERCISE 6 - ROC Curve & AUC
# ==========================================

y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.ylabel('True Positive Rate (Sensitivity)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.legend(loc="lower right")
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.grid(True, alpha=0.3)
plt.show()

"""
INTERPRÉTATION ROC & AUC (A ajouter en Markdown) :
La courbe ROC évalue la performance du classificateur à tous les seuils de décision possibles. La valeur de l'AUC obtenue est généralement très élevée (proche de 0.96+ sur ce type de jeu de données), ce qui démontre une excellente capacité de notre modèle de régression logistique à discriminer correctement les individus atteints de diabète de ceux qui sont sains.
"""